### Favourite Grounds

In [ ]:
mlt.subplots(figsize=(10, 15))
matches['venue'].value_counts().sort_values(ascending=True).plot.bar(
    width=0.8, color=sns.color_palette('inferno', 40)
)
ax.set_xlabel('Grounds')
ax.set_ylabel('count')
mlt.show()

In [ ]:
mlt.subplots(figsize=(10, 6))
ax = matches['player_of_match'].value_counts().head(10).plot.bar(
    width=0.8, color=sns.color_palette('inferno', 10)
)
ax.set_xlabel('player_of_match')
ax.set_ylabel('count')
for p in ax.patches:
    ax.annotate(format(p.get_height()), (p.get_x() + 0.15, p.get_height() + 0.25))
mlt.show()

**Winner by year**

In [ ]:
print('Winner By Years:')
for i in range(2008, 2017):
    df = (matches[matches['season'] == i]).iloc[-1]
    print(df[[1, 10]].values)

**Super Over**

In [ ]:
print('Total Matches with Super Overs:', delivery[delivery['is_super_over'] == 1].match_id.nunique())

In [ ]:
play = list(delivery[delivery['is_super_over'] == 1].batting_team.unique())
print('Teams who haven\'t ever played a super over are: ', list(set(teams) - set(play)))

**Favorite Umpires**

In [ ]:
mlt.subplots(figsize=(10, 6))
ump = pd.concat([matches['umpire1'], matches['umpire2']])
ax = ump.value_counts().head(10).plot.bar(
    width=0.8, color=sns.color_palette('summer', 10)
)
for p in ax.patches:
    ax.annotate(format(p.get_height()), (p.get_x() + 0.15, p.get_height() + 0.25))
mlt.show()

**Team1 vs Team2**

**MI vs KKR**



In [ ]:
def team1_vs_team2(team1, team2):
    mt1 = matches[(matches['team1'] == team1) | (matches['team2'] == team1) & ((matches['team1'] == team2)|(matches['team2'] == team2))]
    sns.countplot(x='season', hue='winner', data=nt1, palette='Set3')
    mlt.xticks(rotation='vertical')
    leg = mlt.legend(loc='upper center')
    fig = mlt.gcf()
    fig.set_size_inches(10, 6)
    mlt.show()

In [ ]:
# Example usage:
team1_vs_team2('MI', 'KKR')
team1_vs_team2('CSK', 'MI')

In [ ]:
def comparator(team1):
    teams = ['MI', 'KKR', 'RCB', 'CSK', 'RR', 'DD', 'SRH', 'KXIP', 'RPS', 'GL', 'PW', 'DC', 'KTK']
    teams.remove(team1)
    opponents = teams.copy()
    mt1 = matches[(matches['team1'] == team1) | (matches['team2'] == team1)]

    for i in opponents:
        mask = ((mt1['team1'] == i) | (mt1['team2'] == i)) & ((mt1['team1'] == team1) | (mt1['team2'] == team1))
        mt2 = mt1.loc[mask, 'winner'].value_counts().to_frame()
        print(f"\n{team1} vs {i}:")
        print(mt2)

In [ ]:
# Example usage:
comparator('MI')
comparator('CSK')

In [ ]:
mlt.subplots(figsize=(12, 6))
xyz = delivery.groupby(['match_id', 'inning', 'batting_team'])['total_runs'].sum().reset_index()
xyz.drop(['match_id'], axis=1, inplace=True)
xyz.sort_values(by=['batting_team', 'total_runs'], ascending=True, inplace=True)

score_1_inning = xyz[xyz['inning'] == 1]
score_2_inning = xyz[xyz['inning'] == 2]

sns.boxplot(x='batting_team', y='total_runs', data=score_1_inning).set_title('1st Innings')
mlt.show()

sns.boxplot(x='batting_team', y='total_runs', data=score_2_inning).set_title('2nd Innings')
fig = mlt.gcf()
fig.set_size_inches(12, 6)
mlt.show()

In [ ]:
high_scores = delivery.groupby(['match_id', 'inning', 'batting_team', 'bowling_team'])['total_runs'].sum().reset_index()
high_scores = high_scores[high_scores['total_runs'] >= 200]
print(high_scores.nlargest(10, 'total_runs'))

In [ ]:
fig, ax = mlt.subplots(1, 2, figsize=(18, 6))
sns.countplot(x=high_scores['batting_team'], ax=ax[0])
ax[0].set_title('Teams Scoring 200+ Runs')
mlt.xticks(rotation=90)

sns.countplot(x=high_scores['bowling_team'], ax=ax[1])
ax[1].set_title('Teams Conceding 200+ Runs')
mlt.xticks(rotation=90)

mlt.tight_layout()
mlt.show()

In [ ]:
all_teams = list(delivery['batting_team'].unique())
teams_scored_200_plus = list(high_scores['batting_team'].unique())
teams_not_scored_200_plus = list(set(all_teams) - set(teams_scored_200_plus))
print(f"Teams who have not scored over 200 runs are: {teams_not_scored_200_plus}")

teams_conceded_200_plus = list(high_scores['bowling_team'].unique())
teams_not_conceded_200_plus = list(set(all_teams) - set(teams_conceded_200_plus))
print(f"Teams who have not conceded over 200 runs are: {teams_not_conceded_200_plus}")

**Chances of Chasing 200+**

In [ ]:
slices = high_scores['is_score_chased'].value_counts()
labels = ['Target Not Chased', 'Target Chased']
colors = ['#1f2ff3', '#fff080']

mlt.pie(slices, labels=labels, colors=colors, startangle=90, shadow=True, explode=(0.1, 0), autopct='%1.1f%%')
fig = mlt.gcf()
fig.set_size_inches(6, 6)
mlt.show()

**Batsman Comparator**

In [ ]:
# 1. Calculate balls faced by each batsman
balls_faced = delivery.groupby('batsman')['ball'].count().reset_index()
balls_faced.rename(columns={'ball': 'balls_faced'}, inplace=True)

# 2. Calculate runs scored by each batsman
runs_scored = delivery.groupby('batsman')['batsman_runs'].sum().reset_index()
runs_scored.rename(columns={'batsman_runs': 'runs_scored'}, inplace=True)

# 3. Merge balls faced and runs scored
batsman_stats = pd.merge(balls_faced, runs_scored, on='batsman', how='outer')

# 4. Calculate fours and sixes
fours = delivery[delivery['batsman_runs'] == 4].groupby('batsman')['batsman_runs'].count().reset_index()
fours.rename(columns={'batsman_runs': 'fours'}, inplace=True)

sixes = delivery[delivery['batsman_runs'] == 6].groupby('batsman')['batsman_runs'].count().reset_index()
sixes.rename(columns={'batsman_runs': 'sixes'}, inplace=True)

# 5. Merge fours and sixes into the main stats DataFrame
batsman_stats = pd.merge(batsman_stats, fours, on='batsman', how='outer')
batsman_stats = pd.merge(batsman_stats, sixes, on='batsman', how='outer')

# Fill NaN values (for players who didn't hit fours/sixes) with 0
batsman_stats[['fours', 'sixes']] = batsman_stats[['fours', 'sixes']].fillna(0).astype(int)

# 6. Calculate strike rate
batsman_stats['strike_rate'] = (batsman_stats['runs_scored'] / batsman_stats['balls_faced']) * 100
batsman_stats['strike_rate'] = batsman_stats['strike_rate'].fillna(0).round(2) # Handle cases with 0 balls faced

# 7. Get highest score and teams played for (more complex aggregation)
# First, get max runs per batsman per match and batting team
max_runs_per_match_team = delivery.groupby(['match_id', 'batsman', 'batting_team'])['batsman_runs'].sum().reset_index()
highest_score_per_batsman = max_runs_per_match_team.groupby('batsman')['batsman_runs'].max().reset_index()
highest_score_per_batsman.rename(columns={'batsman_runs': 'highest_score'}, inplace=True)

# Get all teams a batsman played for
teams_played_for = delivery.groupby('batsman')['batting_team'].unique().apply(list).reset_index()
teams_played_for.rename(columns={'batting_team': 'teams'}, inplace=True)

# Merge highest score and teams into the main stats DataFrame
batsman_stats = pd.merge(batsman_stats, highest_score_per_batsman, on='batsman', how='outer')
batsman_stats = pd.merge(batsman_stats, teams_played_for, on='batsman', how='outer')

print(batsman_stats.head())

In [ ]:
def batsman_comparator(stat1, stat2, batsman1, batsman2, balls_df):
    # Create a FacetGrid for general comparison (assuming 'Team' column exists in balls_df)
    sns.FacetGrid(balls_df, hue='Team', height=8).map(mlt.scatter, stat1, stat2, alpha=0.5).add_legend()

    # Filter and sort data for batsman 1
    bats1 = balls_df[balls_df['batsman'].str.contains(batsman1, na=False)].sort_values(by=stat1, ascending=False)

    # Filter and sort data for batsman 2
    bats2 = balls_df[balls_df['batsman'].str.contains(batsman2, na=False)].sort_values(by=stat1, ascending=False)

    # Plot batsman specific points and annotations
    mlt.scatter(bats1[stat1], bats1[stat2], s=75, c='#FF5733', label=batsman1) # Example color
    mlt.scatter(bats2[stat1], bats2[stat2], s=75, c='#4f73a5', label=batsman2) # Example color

    # Adjust plot size and title
    fig = mlt.gcf()
    fig.set_size_inches(15, 18)
    mlt.title(f'Batsman Comparator: {batsman1} vs {batsman2} ({stat1} vs {stat2})', size=25)
    mlt.legend()
    mlt.show()

In [ ]:
mlt.subplots(figsize=(18, 6))
max_runs = delivery.groupby(['batsman'])['batsman_runs'].sum()

ax = max_runs.sort_values(ascending=False)[:10].plot.bar(width=0.8, color=sns.color_palette('winter_r'))
ax.set_title('Top 10 Batsmen by Total Runs', size=20)
mlt.ylabel('Total Runs')
mlt.xlabel('Batsman')

# Annotate bars with run values
for p in ax.patches:
    ax.annotate(format(p.get_height()), (p.get_x() + 0.1, p.get_height() + 50), fontsize=15)

mlt.show()

In [ ]:
# Reshape the data to get total runs for each type of run (1s, 2s, 3s, 4s, 6s) by batsman
toppers = delivery.groupby(['batsman', 'batsman_runs'])['batsman_runs'].count().unstack(fill_value=0)
toppers.columns = [str(col) + 's' for col in toppers.columns] # Rename columns to '1s', '2s', etc.
# Ensure only relevant columns are present for 1s, 2s, 3s, 4s
toppers = toppers[['1s', '2s', '3s', '4s']].copy() 

fig, ax = mlt.subplots(2, 2, figsize=(18, 12))

# Plot for Most 1s
toppers['1s'].sort_values(ascending=False).head(10).plot(kind='barh', ax=ax[0, 0], color='#45ff45', width=0.8)
ax[0, 0].set_title('Most 1s by Top Batsmen', size=15)
ax[0, 0].set_ylabel('Batsman')
ax[0, 0].set_xlabel('Number of 1s')

# Plot for Most 2s
toppers['2s'].sort_values(ascending=False).head(10).plot(kind='barh', ax=ax[0, 1], color='#a54f73', width=0.8)
ax[0, 1].set_title('Most 2s by Top Batsmen', size=15)
ax[0, 1].set_ylabel('Batsman')
ax[0, 1].set_xlabel('Number of 2s')

# Plot for Most 3s
toppers['3s'].sort_values(ascending=False).head(10).plot(kind='barh', ax=ax[1, 0], color='#4f73a5', width=0.8)
ax[1, 0].set_title('Most 3s by Top Batsmen', size=15)
ax[1, 0].set_ylabel('Batsman')
ax[1, 0].set_xlabel('Number of 3s')

# Plot for Most 4s
toppers['4s'].sort_values(ascending=False).head(10).plot(kind='barh', ax=ax[1, 1], color='#ff4545', width=0.8)
ax[1, 1].set_title('Most 4s by Top Batsmen', size=15)
ax[1, 1].set_ylabel('Batsman')
ax[1, 1].set_xlabel('Number of 4s')

mlt.tight_layout()
mlt.show()

**Top Individual Scores**

In [ ]:
top_scores = delivery.groupby(['match_id', 'batsman', 'batting_team'])['batsman_runs'].sum().reset_index()
top_scores.sort_values('batsman_runs',ascending=0).head(10)
top_scores.nlargest(10, 'batsman_runs')

**Individual Scores by top Batsmen each inning**

In [ ]:
# Assuming 'delivery' DataFrame is loaded
# Define a list of top batsmen for visualization
swam = ['CH Gayle', 'BB McCullum', 'AB de Villiers', 'DA Warner', 'V Kohli', 'MS Dhoni', 'RG Sharma', 'SK Raina', 'G Gambhir', 'RV Uthappa'] # Example list

# Calculate scores per match for each batsman
scores = delivery.groupby(['match_id', 'batsman', 'batting_team'])['batsman_runs'].sum().reset_index()

# Filter scores for batsmen in the 'swam' list
scores_filtered = scores[scores['batsman'].isin(swam)]

mlt.figure(figsize=(14, 8))
sns.swarmplot(x='batsman', y='batsman_runs', data=scores_filtered, hue='batting_team', palette='Set2', size=7)
mlt.ylim(-10, 200) # Set y-axis limits to include lower scores and give context
mlt.title('Individual Scores by Top Batsmen Across Innings', size=16)
mlt.xlabel('Batsman')
mlt.ylabel('Runs Scored')
mlt.legend(title='Batting Team', bbox_to_anchor=(1.05, 1), loc='upper left')
mlt.tight_layout()
mlt.show()

In [ ]:
season_runs = delivery.groupby(['season', 'batsman'])['batsman_runs'].sum().reset_index()

# Find top 5 batsmen by total career runs for plotting
top_5_batsmen = season_runs.groupby('batsman')['batsman_runs'].sum().nlargest(5).index.tolist()

# Filter data for top 5 batsmen
season_runs_filtered = season_runs[season_runs['batsman'].isin(top_5_batsmen)]

# Pivot the data for plotting: seasons as index, batsmen as columns, runs as values
plot_data = season_runs_filtered.pivot_table(index='season', columns='batsman', values='batsman_runs', fill_value=0)

# Plotting
mlt.figure(figsize=(16, 6))
plot_data.plot(kind='line', marker='o', ax=mlt.gca(), linewidth=2)
mlt.title('Runs Scored by Top Batsmen Across Seasons', size=20)
mlt.xlabel('Season')
mlt.ylabel('Total Runs')
mlt.xticks(rotation=45)
mlt.grid(True, linestyle='--', alpha=0.6)
mlt.legend(title='Batsman', bbox_to_anchor=(1.05, 1), loc='upper left')
mlt.tight_layout()
mlt.show()

**Frequency of Scores**

In [ ]:
mlt.figure(figsize=(10, 6))
bins = np.arange(0, top_scores['batsman_runs'].max() + 10, 10) # Create bins from 0 to max score with interval 10
mlt.hist(top_scores['batsman_runs'], bins=bins, histtype='bar', rwidth=0.9, color='#0FFFFF', edgecolor='black')
mlt.xlabel('Runs Scored (Bins)')
mlt.ylabel('Count of Scores')
mlt.title('Frequency Distribution of Batsman Scores', size=16)

# Calculate and plot the mean score
mean_score = top_scores['batsman_runs'].mean()
mlt.axvline(mean_score, color='r', linestyle='dashed', linewidth=2, label=f'Average Score: {mean_score:.2f}')
mlt.legend()
mlt.grid(axis='y', alpha=0.75)
mlt.show()

**Orange caps each season (Highest run getter per season)**

In [ ]:
# Merge matches and delivery dataframes to get season information for each ball
orange_cap_data = pd.merge(delivery, matches[['id', 'season']], left_on='match_id', right_on='id', how='left')

# Group by season and batsman to get total runs per batsman per season
season_batsman_runs = orange_cap_data.groupby(['season', 'batsman'])['batsman_runs'].sum().reset_index()

# Sort to find the highest run-getter for each season
season_batsman_runs_sorted = season_batsman_runs.sort_values(by=['season', 'batsman_runs'], ascending=[True, False])

# Drop duplicates to keep only the top scorer for each season (Orange Cap holder)
orange_cap_holders = season_batsman_runs_sorted.drop_duplicates(subset=['season'], keep='first')

# Create Plotly bar chart
trace1 = go.Bar(
    x=orange_cap_holders['season'].values,
    y=orange_cap_holders['batsman_runs'].values,
    name='Orange Cap Holders',
    marker=dict(
        color='rgb(255,140,0)', # Orange color
        line=dict(color='rgb(8,48,107)', width=1.5)
    ),
    opacity=1,
    text=orange_cap_holders['batsman'], # Show batsman name on hover
    hoverinfo='text+y'
)

layout = go.Layout(
    title='Orange Cap Holders Each Season',
    xaxis=dict(title='Season'),
    yaxis=dict(title='Total Runs'),
    hovermode='closest'
)

fig = go.Figure(data=[trace1], layout=layout)
fig.show()

### Top Bowlers

**Highest Wicket Taker**

In [ ]:
# Assuming 'delivery' DataFrame is loaded
dismissal_kinds = ['bowled', 'caught', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']

# Filter deliveries for actual wickets (excluding run-outs, retirements, etc.)
ct = delivery[delivery['dismissal_kind'].isin(dismissal_kinds)]

# Get top 10 bowlers by wicket count
ax = ct['bowler'].value_counts()[1:10].plot.bar(width=0.8, color=sns.color_palette('summer_r', 20))

# Annotate bars with wicket counts
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=15)

mlt.title('Top 10 Wicket-Takers (Excluding 0-wicket dismissals)', size=20)
mlt.xlabel('Bowler')
mlt.ylabel('Number of Wickets')
mlt.tight_layout()
mlt.show()

**Maximum Overs**

In [ ]:
# Group by bowler to get total runs given and total balls bowled
eco = delivery.groupby(['bowler']).sum(numeric_only=True).reset_index()
eco['total_balls'] = delivery['bowler'].value_counts().reset_index(name='count')['count'] # Correctly get total balls
eco['overs'] = eco['total_balls'] // 6 # Calculate overs bowled (integer division)

# Filter for bowlers with more than 200 overs and display top 5
print("\nTop 5 Bowlers by Overs Bowled (min 200 overs):")
print(eco[eco['overs'] > 200].sort_values(by='overs', ascending=False).head(5)[['bowler', 'overs']])

# Calculate economy rate
eco['economy'] = eco['total_runs'] / eco['overs']

# Filter for bowlers with more than 300 overs and display top 10 most economical
print("\nTop 10 Most Economical Bowlers (min 300 overs):")
print(eco[eco['overs'] > 300].sort_values(by='economy').head(10)[['bowler', 'economy']])

**Top 20 Bowlers**

In [ ]:
# Step 1: Get total runs given by each bowler
bowlers_runs = delivery.groupby('bowler')['batsman_runs'].sum().reset_index()
bowlers_runs.rename(columns={'batsman_runs': 'runs_given'}, inplace=True)

# Step 2: Get total balls bowled by each bowler
bowlers_balls = delivery['bowler'].value_counts().reset_index()
bowlers_balls.columns = ['bowler', 'balls']

# Step 3: Merge runs and balls data
bowlers_stats = pd.merge(bowlers_runs, bowlers_balls, on='bowler', how='left')
bowlers_stats['overs'] = bowlers_stats['balls'] // 6

# Step 4: Get wickets taken by each bowler (excluding run-outs, etc.)
dismissal_kinds = ['bowled', 'caught', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']
wickets_taken = delivery[delivery['dismissal_kind'].isin(dismissal_kinds)]
wickets_count = wickets_taken['bowler'].value_counts().reset_index()
wickets_count.columns = ['bowler', 'wickets']

# Step 5: Merge wickets data
bowlers_stats = pd.merge(bowlers_stats, wickets_count, on='bowler', how='left')
bowlers_stats['wickets'].fillna(0, inplace=True) # Fill NaN wickets with 0
bowlers_stats['wickets'] = bowlers_stats['wickets'].astype(int)

# Step 6: Calculate economy rate
bowlers_stats['economy'] = bowlers_stats['runs_given'] / bowlers_stats['overs']

# Filter for top 20 bowlers (e.g., by wickets or overs, here by wickets for demonstration)
# And ensure overs are not zero to avoid division by zero for economy
final_bowlers_df = bowlers_stats[bowlers_stats['overs'] > 0].sort_values(by='wickets', ascending=False).head(20)

print("\nTop 20 Bowlers with Runs Given, Balls, Overs, Wickets, and Economy:")
print(final_bowlers_df.reset_index(drop=True))

In [ ]:
bowlers = final_bowlers_df.head(20).reset_index(drop=True)

trace = go.Scatter(
    x=bowlers['bowler'].values,
    y=bowlers['wickets'].values,
    mode='markers',
    marker=dict(
        size=bowlers['wickets'].values * 0.8, # Adjust size based on wickets for better visualization
        color=bowlers['economy'].values,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Economy Rate')
    ),
    text=bowlers['overs'].values, # Show overs on hover
    hoverinfo='x+y+text'
)

layout = go.Layout(
    autosize=True,
    hovermode='closest',
    showgrid=False,
    zeroline=False,
    showline=False,
    title='Top 20 Wicket-Taking Bowlers (Size by Wickets, Color by Economy)',
    xaxis=dict(title='Bowler', ticklen=5, gridwidth=2, zeroline=False, showline=False),
    yaxis=dict(title='Wickets Taken', ticklen=5, gridwidth=2, zeroline=False, showline=False),
    showlegend=False
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [ ]:
def get_top_bowler(batsman_name, delivery_df):
    # Filter deliveries for the specific batsman
    batsman_data = delivery_df[delivery_df['batsman'] == batsman_name].copy()

    # Define dismissal kinds that count as wickets
    dismissal_kinds = ['caught', 'lbw', 'bowled', 'stumped', 'caught and bowled', 'hit wicket']

    # Filter for actual dismissals against this batsman
    batsman_dismissals = batsman_data[batsman_data['dismissal_kind'].isin(dismissal_kinds)]

    # Group by bowler and count dismissals, then sort
    if not batsman_dismissals.empty:
        top_bowler_data = batsman_dismissals.groupby('bowler')['dismissal_kind'].count().reset_index()
        top_bowler_data.rename(columns={'dismissal_kind': 'No_of_Dismissals'}, inplace=True)
        top_bowler = top_bowler_data.sort_values(by='No_of_Dismissals', ascending=False).iloc[0]
        
        # Add batsman's name to the result
        top_bowler_df = pd.DataFrame([{'batsman': batsman_name, 'bowler': top_bowler['bowler'], 'No_of_Dismissals': top_bowler['No_of_Dismissals']}])
        return top_bowler_df
    else:
        return pd.DataFrame([{'batsman': batsman_name, 'bowler': 'N/A', 'No_of_Dismissals': 0}])

**Highest Dismissals for Multiple Batsmen**

In [ ]:
# List of batsmen to analyze
batsmen_to_analyze = ['CH Gayle', 'V Kohli', 'SK Raina', 'MS Dhoni', 'RG Sharma'] # Example list

# Collect results for each batsman
results = []
for batsman in batsmen_to_analyze:
    results.append(get_top_bowler(batsman, delivery))

# Concatenate all individual results into a single DataFrame
new_df = pd.concat(results, ignore_index=True)

# Display the final DataFrame
print("\nHighest Dismissals for Batsmen by a Single Bowler:")
print(new_df)

**Purple Cap Holders Each Season**

In [ ]:
# Define dismissal kinds that count as actual wickets
dismissal_kinds = ['bowled', 'caught', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']

# Filter delivery data for actual wickets
wickets_data = delivery[delivery['dismissal_kind'].isin(dismissal_kinds)].copy()

# Merge with matches DataFrame to get season information
purple_cap_data = pd.merge(wickets_data, matches[['id', 'season']], left_on='match_id', right_on='id', how='left')

# Group by season and bowler to count wickets
purple = purple_cap_data.groupby(['season', 'bowler'])['dismissal_kind'].count().reset_index()

# Sort by wickets in descending order within each season
purple = purple.sort_values(by=['season', 'dismissal_kind'], ascending=[True, False])

# Drop duplicates to keep only the top wicket-taker (Purple Cap holder) for each season
purple_cap_holders = purple.drop_duplicates(subset=['season'], keep='first').sort_values(by='season')

# Rename 'dismissal_kind' column to 'count_wickets' for clarity
purple_cap_holders.rename(columns={'dismissal_kind': 'count_wickets'}, inplace=True)

# Create Plotly bar chart
trace1 = go.Bar(
    x=purple_cap_holders['season'].values,
    y=purple_cap_holders['count_wickets'].values,
    name='Purple Cap Holders',
    marker=dict(
        color='rgb(75,0,130)', # Indigo/Purple color
        line=dict(color='rgb(8,48,107)', width=1.5)
    ),
    opacity=1,
    text=purple_cap_holders['bowler'], # Show bowler name on hover
    hoverinfo='text+y'
)

layout = go.Layout(
    title='Purple Cap Holders Each Season',
    xaxis=dict(title='Season'),
    yaxis=dict(title='Total Wickets'),
    hovermode='closest'
)

fig = go.Figure(data=[trace1], layout=layout)
fig.show()